In [28]:
import time

import os
import cv2
import torch
from ultralytics import YOLO

In [29]:
from code_programm.path import get_path_weight_model

In [30]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())

1
NVIDIA GeForce GTX 1080 Ti


In [40]:
# Папка с изображениями
folder_path = r'D:\Dataset_for_autopilot\numbers'

# Получаем список файлов в папке
files = os.listdir(folder_path)

# Фильтруем файлы, оставляем только изображения
image_files = [file for file in files if file.endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model_line = YOLO(get_path_weight_model('best.pt'))
data_time = []
data = [0, 0]
# Проходимся по каждому изображению и читаем его
for image_file in image_files:
    image_path = os.path.join(folder_path, image_file)
    image = cv2.imread(image_path)
    # Делаем что-то с изображением, например, показываем его
    cv2.imshow('Image', image)
    start = time.time()
    # image = cv2.resize(image, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    results = model_line.predict(image, conf=0.85, show=True, device='cuda')
    sorted_objects = sorted(
        ({'class': int(cls), 'confidence': float(conf), 'xmin': int(xmin), 'ymin': int(ymin), 'xmax': int(xmax),
          'ymax': int(ymax)}
         for result in results for obj in result.boxes.data for xmin, ymin, xmax, ymax, conf, cls in (obj.tolist(),)),
        key=lambda obj: obj['xmin']
    )

    if sorted_objects:
        speed = ''.join(str(obj['class']) for obj in sorted_objects)
    else:
        speed = ''
    name = image_file.replace('.png', '')
    times = start - time.time()
    print(times)
    data_time.append(times)
    print(speed, ' ||||||| ', name)
    if str(speed) == str(name):
        print('TRUE +++++++++++++++')
        data[0] += 1
    else:
        print('FALSE --------------')
        data[1] += 1
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cv2.waitKey()
cv2.destroyAllWindows()
print(sum(data_time) / len(data_time))
print(data)

1
NVIDIA GeForce GTX 1080 Ti
0: 544x640 1 0, 8.1ms
Speed: 3.0ms preprocess, 8.1ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 640)
-0.23314595222473145
0  |||||||  0
TRUE +++++++++++++++

0: 544x640 1 1, 10.0ms
Speed: 3.0ms preprocess, 10.0ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 640)
-0.025999069213867188
1  |||||||  1
TRUE +++++++++++++++

0: 544x640 1 0, 1 1, 9.0ms
Speed: 3.0ms preprocess, 9.0ms inference, 3.0ms postprocess per image at shape (1, 3, 544, 640)
-0.034003257751464844
10  |||||||  10
TRUE +++++++++++++++
0: 544x640 2 0s, 1 1, 7.0ms
Speed: 3.0ms preprocess, 7.0ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 640)
-0.03400063514709473
100  |||||||  100
TRUE +++++++++++++++
0: 544x640 1 0, 2 1s, 8.0ms
Speed: 3.0ms preprocess, 8.0ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 640)
-0.03400158882141113
101  |||||||  101
TRUE +++++++++++++++
0: 544x640 1 0, 1 1, 1 2, 7.0ms
Speed: 3.0ms preprocess, 7.0ms infer

In [33]:
cv2.waitKey()
cv2.destroyAllWindows()